In [1]:
from math import gcd


def extended_gcd_verbose(a: int, b: int) -> tuple[int, int, int]:
    """
    拡張ユークリッドの互除法を途中表示する。
    戻り値:
        gcd(a, b), x, y
    ax + by = gcd(a, b)
    """
    original_a = a
    original_b = b

    old_r, r = a, b
    old_s, s = 1, 0
    old_t, t = 0, 1

    print("\n拡張ユークリッド互除法")
    print("----------------------------------------")
    print(" q      old_r      r      old_s      s      old_t      t")
    print(
        f"{'-':>2} {old_r:>10} {r:>6}"
        f" {old_s:>10} {s:>6} {old_t:>10} {t:>6}"
    )

    while r != 0:
        quotient = old_r // r

        old_r, r = r, old_r - quotient * r
        old_s, s = s, old_s - quotient * s
        old_t, t = t, old_t - quotient * t

        print(
            f"{quotient:>2} {old_r:>10} {r:>6}"
            f" {old_s:>10} {s:>6} {old_t:>10} {t:>6}"
        )

    print("----------------------------------------")
    print(f"gcd({original_a}, {original_b}) = {old_r}")
    print(
        f"{original_a} × ({old_s}) "
        f"+ {original_b} × ({old_t}) = {old_r}"
    )

    return old_r, old_s, old_t


def modular_inverse_verbose(e: int, phi: int) -> int:
    """
    eのmod phiにおける逆元dを途中表示して求める。
    """
    print("\n秘密指数 d を求めます")
    print(f"条件: {e} × d ≡ 1 mod {phi}")

    g, x, _ = extended_gcd_verbose(e, phi)

    if g != 1:
        raise ValueError(
            "逆元が存在しません。eとφ(n)は互いに素である必要があります。"
        )

    d = x % phi

    print(f"\n得られた係数 x = {x}")
    print(f"d = {x} mod {phi}")
    print(f"d = {d}")
    print(f"確認: {e} × {d} mod {phi} = {(e * d) % phi}")

    return d


def modular_power_verbose(
    base: int,
    exponent: int,
    modulus: int,
    title: str
) -> int:
    """
    繰り返し二乗法により base^exponent mod modulus を計算し、
    途中経過を表示する。
    """
    if exponent < 0:
        raise ValueError("指数は0以上にしてください。")

    result = 1
    current_base = base % modulus
    current_exponent = exponent
    step = 1

    print(f"\n{title}")
    print("----------------------------------------")
    print(f"計算: {base}^{exponent} mod {modulus}")
    print(f"指数 {exponent} の2進数: {exponent:b}")
    print()
    print(
        f"{'回':>3} {'指数':>8} {'最下位ビット':>14}"
        f" {'result':>10} {'base':>10}"
    )

    while current_exponent > 0:
        bit = current_exponent % 2

        print(
            f"{step:>3} {current_exponent:>8} {bit:>14}"
            f" {result:>10} {current_base:>10}"
        )

        if bit == 1:
            old_result = result
            result = (result * current_base) % modulus

            print(
                f"    ビットが1なので resultを更新:"
                f" ({old_result} × {current_base}) mod {modulus}"
                f" = {result}"
            )
        else:
            print("    ビットが0なので resultは更新しません。")

        old_base = current_base
        current_base = (current_base * current_base) % modulus

        print(
            f"    baseを二乗:"
            f" ({old_base} × {old_base}) mod {modulus}"
            f" = {current_base}"
        )

        current_exponent //= 2
        print(f"    指数を2で割る: 次の指数 = {current_exponent}")
        print()

        step += 1

    print(f"計算結果: {base}^{exponent} mod {modulus} = {result}")

    return result


def main() -> None:
    print("=" * 60)
    print("RSA暗号の途中経過表示プログラム")
    print("=" * 60)

    # 1. 素数を選ぶ
    p = 5
    q = 11

    print("\n1. 2つの素数を選びます")
    print(f"p = {p}")
    print(f"q = {q}")

    # 2. nを計算
    n = p * q

    print("\n2. nを計算します")
    print(f"n = p × q")
    print(f"n = {p} × {q}")
    print(f"n = {n}")

    # 3. オイラー関数
    phi = (p - 1) * (q - 1)

    print("\n3. オイラー関数 φ(n) を計算します")
    print("φ(n) = (p - 1) × (q - 1)")
    print(f"φ(n) = ({p} - 1) × ({q} - 1)")
    print(f"φ(n) = {p - 1} × {q - 1}")
    print(f"φ(n) = {phi}")

    # 4. 公開指数
    e = 3

    print("\n4. 公開指数 e を選びます")
    print(f"e = {e}")
    print(f"gcd({e}, {phi}) = {gcd(e, phi)}")

    if gcd(e, phi) != 1:
        raise ValueError("eとφ(n)は互いに素である必要があります。")

    # 5. 秘密指数
    d = modular_inverse_verbose(e, phi)

    print("\n5. 鍵が完成しました")
    print(f"公開鍵: (e, n) = ({e}, {n})")
    print(f"秘密鍵: (d, n) = ({d}, {n})")

    # 平文
    message = 9

    if not 0 <= message < n:
        raise ValueError(f"平文は0以上{n - 1}以下にしてください。")

    print("\n6. 平文を設定します")
    print(f"M = {message}")

    # 暗号化
    ciphertext = modular_power_verbose(
        base=message,
        exponent=e,
        modulus=n,
        title="7. 公開鍵による暗号化"
    )

    print("\n暗号化の式")
    print("C = M^e mod n")
    print(f"C = {message}^{e} mod {n}")
    print(f"C = {ciphertext}")

    # 復号
    decrypted_message = modular_power_verbose(
        base=ciphertext,
        exponent=d,
        modulus=n,
        title="8. 秘密鍵による復号"
    )

    print("\n復号の式")
    print("M = C^d mod n")
    print(f"M = {ciphertext}^{d} mod {n}")
    print(f"M = {decrypted_message}")

    # 最終結果
    print("\n" + "=" * 60)
    print("最終結果")
    print("=" * 60)
    print(f"元の平文       : {message}")
    print(f"暗号文         : {ciphertext}")
    print(f"復号後の平文   : {decrypted_message}")

    if message == decrypted_message:
        print("判定           : 復号成功")
    else:
        print("判定           : 復号失敗")


if __name__ == "__main__":
    main()

RSA暗号の途中経過表示プログラム

1. 2つの素数を選びます
p = 5
q = 11

2. nを計算します
n = p × q
n = 5 × 11
n = 55

3. オイラー関数 φ(n) を計算します
φ(n) = (p - 1) × (q - 1)
φ(n) = (5 - 1) × (11 - 1)
φ(n) = 4 × 10
φ(n) = 40

4. 公開指数 e を選びます
e = 3
gcd(3, 40) = 1

秘密指数 d を求めます
条件: 3 × d ≡ 1 mod 40

拡張ユークリッド互除法
----------------------------------------
 q      old_r      r      old_s      s      old_t      t
 -          3     40          1      0          0      1
 0         40      3          0      1          1      0
13          3      1          1    -13          0      1
 3          1      0        -13     40          1     -3
----------------------------------------
gcd(3, 40) = 1
3 × (-13) + 40 × (1) = 1

得られた係数 x = -13
d = -13 mod 40
d = 27
確認: 3 × 27 mod 40 = 1

5. 鍵が完成しました
公開鍵: (e, n) = (3, 55)
秘密鍵: (d, n) = (27, 55)

6. 平文を設定します
M = 9

7. 公開鍵による暗号化
----------------------------------------
計算: 9^3 mod 55
指数 3 の2進数: 11

  回       指数         最下位ビット     result       base
  1        3              1          1          9
 